In [ ]:
!pip install -q openai-whisper pyannote.audio transformers
!pip install -q scipy pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 48.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 893.7/893.7 kB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 6.5 MB/s 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CORPUS_C = '/content/drive/MyDrive/CHF_Corpus/corpus_c'
print("Archivos MP3 en corpus_c:")
for f in sorted(os.listdir(CORPUS_C)):
    if f.endswith('.mp3'):
        print(f"  {f}")

Mounted at /content/drive
Archivos MP3 en corpus_c:
  Audiencia de Observaciones a la Resolución de Conclusiones No. 05 del Caso 03 ｜ 20260423.mp3
  Audiencia de Observaciones a la Resolución de Conclusiones No. 05 del Caso 03 ｜ 20260424.mp3
  Audiencia de Observaciones de Víctimas - Fase nacional del Caso 03 ｜ 20260409.mp3
  Audiencia de Reconocimiento de Verdad ｜ Caso 03 ｜ Subcaso Casanare ｜ 20230918.mp3
  Audiencia de Reconocimiento y Aceptación de Responsabilidad – Caso 03 - Subcaso Huila ｜ 20240810.mp3
  Audiencia de observaciones a versiones de militares en el Caso 03 ('falsos positivos'), Subcaso Meta.mp3
  Audiencia de observaciones de víctimas acreditadas Caso 03 ｜ Subcaso Costa Caribe ｜ Atanquez, Cesar.mp3
  Caso 03： Audiencia de Reconocimiento por 'falsos positivos' en el Catatumbo.mp3
  Caso 03｜ Audiencia de observaciones de las víctimas｜ Subcaso Huila (17 de mayo de 2022).mp3
  Día 1 Audiencia de observaciones de víctimas ｜ Subcaso Antioquia ｜ Caso 03 ｜ 20240301.mp

In [ ]:
import os

CORPUS_C = '/content/drive/MyDrive/CHF_Corpus/corpus_c'

# Los 5 ya procesados con Whisper
ya_procesados = {
    'casanare_torres.mp3',
    'costa_caribe.mp3',
    'dabeiba_antioquia.mp3',
    'huila.mp3',
    'Caso 03： Audiencia de Reconocimiento por \'falsos positivos\' en el Catatumbo.mp3',
}

mp3_nuevos = []
for f in sorted(os.listdir(CORPUS_C)):
    if f.endswith('.mp3') and f not in ya_procesados:
        mp3_nuevos.append(f)
        print(f"  NUEVO: {f}")

print(f"\nTotal MP3 nuevos: {len(mp3_nuevos)}")

  NUEVO: Audiencia de Observaciones a la Resolución de Conclusiones No. 05 del Caso 03 ｜ 20260423.mp3
  NUEVO: Audiencia de Observaciones a la Resolución de Conclusiones No. 05 del Caso 03 ｜ 20260424.mp3
  NUEVO: Audiencia de Observaciones de Víctimas - Fase nacional del Caso 03 ｜ 20260409.mp3
  NUEVO: Audiencia de Reconocimiento de Verdad ｜ Caso 03 ｜ Subcaso Casanare ｜ 20230918.mp3
  NUEVO: Audiencia de Reconocimiento y Aceptación de Responsabilidad – Caso 03 - Subcaso Huila ｜ 20240810.mp3
  NUEVO: Audiencia de observaciones a versiones de militares en el Caso 03 ('falsos positivos'), Subcaso Meta.mp3
  NUEVO: Audiencia de observaciones de víctimas acreditadas Caso 03 ｜ Subcaso Costa Caribe ｜ Atanquez, Cesar.mp3
  NUEVO: Caso 03｜ Audiencia de observaciones de las víctimas｜ Subcaso Huila (17 de mayo de 2022).mp3
  NUEVO: Día 1 Audiencia de observaciones de víctimas ｜ Subcaso Antioquia ｜ Caso 03 ｜ 20240301.mp3
  NUEVO: Día 2 ｜ Tercera audiencia de observaciones de víctimas ｜ C

In [ ]:
import whisper
import json
import os

CORPUS_C = '/content/drive/MyDrive/CHF_Corpus/corpus_c'

model = whisper.load_model("large-v3")
print("✓ Whisper large-v3 cargado")

for mp3_file in mp3_nuevos:
    ruta = f"{CORPUS_C}/{mp3_file}"
    nombre = mp3_file.replace('.mp3', '')
    txt_path = f"{CORPUS_C}/{nombre}.txt"
    json_path = f"{CORPUS_C}/{nombre}_segments.json"

    # Saltar si ya existe
    if os.path.exists(txt_path):
        print(f"  ya existe: {mp3_file}")
        continue

    print(f"\nTranscribiendo: {mp3_file[:60]}...")
    result = model.transcribe(ruta, language="es", verbose=False)

    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write(result['text'])

    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(result['segments'], f, ensure_ascii=False, indent=2)

    print(f"  ✓ {len(result['segments'])} segmentos guardados")


print("\n✓ Todas las transcripciones completadas")

ModuleNotFoundError: No module named 'whisper'

In [ ]:
import os
CORPUS_C = '/content/drive/MyDrive/CHF_Corpus/corpus_c'
txts = [f for f in os.listdir(CORPUS_C) if f.endswith('.txt')]
print(f"TXT existentes: {len(txts)}")
for t in sorted(txts):
    print(f"  {t}")

TXT existentes: 8
  Audiencia de Observaciones a la Resolución de Conclusiones No. 05 del Caso 03 ｜ 20260423.txt
  README.txt
  casanare_torres.txt
  catatumbo.txt
  catatumbo_audiencia_reconocimiento.txt
  costa_caribe.txt
  dabeiba_antioquia.txt
  huila.txt


In [ ]:
import whisper
import json
import os

CORPUS_C = '/content/drive/MyDrive/CHF_Corpus/corpus_c'
model = whisper.load_model("large-v3")
print("✓ Whisper cargado")

for mp3_file in mp3_nuevos:
    nombre = mp3_file.replace('.mp3', '')
    txt_path = f"{CORPUS_C}/{nombre}.txt"
    json_path = f"{CORPUS_C}/{nombre}_segments.json"

    if os.path.exists(txt_path):
        print(f"  ✓ ya existe: {mp3_file[:50]}")
        continue

    print(f"\n▶ Transcribiendo: {mp3_file[:60]}...")
    try:
        result = model.transcribe(
            f"{CORPUS_C}/{mp3_file}",
            language="es",
            verbose=False,
            fp16=True,
            condition_on_previous_text=False
        )
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(result['text'])
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(result['segments'], f, ensure_ascii=False, indent=2)
        print(f"  ✓ {len(result['segments'])} segmentos | guardado en Drive")
    except Exception as e:
        print(f"  ✗ Error: {e}")
        continue

print("\n✓ Proceso completado")

100%|██████████████████████████████████████| 2.88G/2.88G [00:11<00:00, 260MiB/s]


✓ Whisper cargado
  ✓ ya existe: Audiencia de Observaciones a la Resolución de Con

▶ Transcribiendo: Audiencia de Observaciones a la Resolución de Conclusiones ...


100%|██████████| 2494000/2494000 [39:05<00:00, 1063.40frames/s]


  ✓ 2830 segmentos | guardado en Drive

▶ Transcribiendo: Audiencia de Observaciones de Víctimas - Fase nacional del ...


100%|██████████| 3122501/3122501 [53:02<00:00, 981.14frames/s] 


...

  ✓ 3000 segmentos | guardado en Drive

▶ Transcribiendo: Día 2 ｜ Tercera audiencia de observaciones de víctimas ｜ C...


100%|██████████| 2024998/2024998 [39:49<00:00, 847.46frames/s] 


  ✓ 3724 segmentos | guardado en Drive

▶ Transcribiendo: Segunda audiencia de observaciones de víctimas ｜ Caso 03 ｜ ...


 99%|█████████▉| 3105464/3123464 [49:21<00:17, 1048.61frames/s]

  ✓ 4080 segmentos | guardado en Drive

✓ Proceso completado


In [1]:
# CELDA 1 - reinstalar
!pip install -q openai-whisper transformers torch scipy pandas numpy

# CELDA 2 - montar Drive
from google.colab import drive
drive.mount('/content/drive')
import os
CORPUS_C = '/content/drive/MyDrive/CHF_Corpus/corpus_c'

# CELDA 3 - redefinir mp3_nuevos (para la celda 5)
mp3_nuevos = [f for f in sorted(os.listdir(CORPUS_C))
              if f.endswith('.mp3') and f not in {
                  'casanare_torres.mp3', 'costa_caribe.mp3',
                  'dabeiba_antioquia.mp3', 'huila.mp3',
                  "Caso 03： Audiencia de Reconocimiento por 'falsos positivos' en el Catatumbo.mp3"
              }]
print(f"MP3 nuevos: {len(mp3_nuevos)}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 3.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 1.5 MB/s eta 0:00:00
Mounted at /content/drive
MP3 nuevos: 11


In [2]:
!pip install -q transformers torch scipy pandas numpy

In [3]:
import json, os

CORPUS_C = '/content/drive/MyDrive/CHF_Corpus/corpus_c'

LEXICON = [
    "mi hijo", "mi hija", "mi hermano", "mi hermana", "mi madre", "mi padre",
    "mi esposo", "mi esposa", "nuestro hijo", "nuestros hijos",
    "nos mataron", "lo mataron", "la mataron", "lo asesinaron",
    "nunca volvió", "nunca regresó", "era inocente", "era civil",
    "no era guerrillero", "soy la mamá", "soy la madre", "soy el padre",
    "soy hermana", "soy hermano", "vengo por mi",
    "dolor", "sufrimiento", "llorar", "inocente", "civil", "campesino",
    "buscando justicia", "buscando verdad", "quiero saber",
]

INSTITUCIONAL = [
    "sala de reconocimiento", "jurisdicción especial", "compareciente",
    "magistrada", "magistrado", "subcaso", "macrocaso", "apoderado judicial",
]

def score(texto):
    t = texto.lower()
    return max(0, sum(1 for f in LEXICON if f in t) - sum(1 for f in INSTITUCIONAL if f in t))

todos = []
for f in sorted(os.listdir(CORPUS_C)):
    if not f.endswith('_segments.json'):
        continue
    audiencia = f.replace('_segments.json', '')
    with open(f"{CORPUS_C}/{f}", encoding='utf-8') as fh:
        segs = json.load(fh)
    candidatos = []
    for s in segs:
        txt = s.get('text','').strip() if isinstance(s,dict) else str(s)
        sc = score(txt)
        if sc >= 2 and len(txt) >= 40:
            candidatos.append({'texto': txt, 'audiencia': audiencia, 'score': sc})
    mejores = sorted(candidatos, key=lambda x: x['score'], reverse=True)[:40]
    todos.extend(mejores)
    print(f"  {audiencia[:50]}: {len(candidatos)} → {len(mejores)}")

print(f"\nTotal candidatos: {len(todos)}")
with open(f"{CORPUS_C}/candidatos_victimas_v4.json", 'w', encoding='utf-8') as f:
    json.dump({'total': len(todos), 'segmentos': todos}, f, ensure_ascii=False, indent=2)
print("✓ Guardado: candidatos_victimas_v4.json")

  Audiencia de Observaciones a la Resolución de Con: 2 → 2
  Audiencia de Observaciones a la Resolución de Con: 3 → 3
  Audiencia de Observaciones de Víctimas - Fase nac: 18 → 18
  Audiencia de Reconocimiento de Verdad ｜ Caso 03 ｜ : 9 → 9
  Audiencia de Reconocimiento y Aceptación de Respo: 6 → 6
  Audiencia de observaciones a versiones de militare: 5 → 5
  Audiencia de observaciones de víctimas acreditada: 2 → 2
  Caso 03｜ Audiencia de observaciones de las víctim: 10 → 10
  Día 1 Audiencia de observaciones de víctimas ｜ S: 5 → 5
  Día 2 ｜ Tercera audiencia de observaciones de ví: 8 → 8
  Segunda audiencia de observaciones de víctimas ｜ : 2 → 2
  casanare_torres: 0 → 0
  catatumbo_audiencia_reconocimiento: 9 → 9
  costa_caribe: 10 → 10
  dabeiba_antioquia: 7 → 7
  huila: 6 → 6

Total candidatos: 102
✓ Guardado: candidatos_victimas_v4.json


In [5]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, numpy as np
CORPUS_C = '/content/drive/MyDrive/CHF_Corpus/corpus_c'
REF_DIR = '/content/drive/MyDrive/CHF_Corpus/referencias'
print("✓ Drive montado")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Drive montado


In [7]:
# Cargar centroide v3b existente
centroide_v3b = np.load(f"{REF_DIR}/centroide_mafapo_v3b.npy")
print(f"✓ Centroide v3b cargado — norma={np.linalg.norm(centroide_v3b):.3f}")

# Cargar 102 candidatos
with open(f"{CORPUS_C}/candidatos_victimas_v4.json", encoding='utf-8') as f:
    data = json.load(f)
textos_nuevos = [s['texto'] for s in data['segmentos']]
print(f"✓ {len(textos_nuevos)} candidatos cargados")

✓ Centroide v3b cargado — norma=14.558
✓ 102 candidatos cargados


In [8]:
import torch
from transformers import AutoTokenizer, AutoModel
from scipy.spatial.distance import cosine as cosine_dist

DEVICE = torch.device('cpu')
MODEL = "eventdata-utd/ConfliBERT-Spanish-Beto-Cased-v1"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL).to(DEVICE)
model.eval()
print("✓ ConfliBERT cargado en CPU")

def get_emb(texto):
    inp = tokenizer(texto, return_tensors="pt", max_length=512,
                    truncation=True, padding=True)
    with torch.no_grad():
        out = model(**inp)
    return out.last_hidden_state[:, 0, :].squeeze().numpy()

print("Calculando embeddings de 102 candidatos...")
embs_nuevos = []
for i, t in enumerate(textos_nuevos):
    embs_nuevos.append(get_emb(t))
    if (i+1) % 10 == 0:
        print(f"  {i+1}/102...")

vec_v3b = centroide_v3b * 67
vec_nuevos = np.sum([e * 1.8 for e in embs_nuevos], axis=0)
n_pond = 67 + len(embs_nuevos) * 1.8
centroide_v4 = (vec_v3b + vec_nuevos) / n_pond
centroide_v4 = centroide_v4 / np.linalg.norm(centroide_v4)

dist = cosine_dist(centroide_v3b, centroide_v4)
n_ef = 67 + len(embs_nuevos)
margen = 100 / np.sqrt(n_ef)

print(f"\n{'='*50}")
print(f"CENTROIDE MAFAPO v4")
print(f"  Total textos:  {n_ef}")
print(f"  Margen error:  ±{margen:.1f}%")
print(f"  Dist v3b→v4:   {dist:.4f}")
print(f"  {'✓ SATURADO' if dist < 0.005 else '— variación presente'}")

np.save(f"{REF_DIR}/centroide_mafapo_v4.npy", centroide_v4)
print(f"✓ Centroide v4 guardado")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/729k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: eventdata-utd/ConfliBERT-Spanish-Beto-Cased-v1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ ConfliBERT cargado en CPU
Calculando embeddings de 102 candidatos...


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

  10/102...
  20/102...
  30/102...
  40/102...
  50/102...
  60/102...
  70/102...
  80/102...
  90/102...
  100/102...

CENTROIDE MAFAPO v4
  Total textos:  169
  Margen error:  ±7.7%
  Dist v3b→v4:   0.0066
  — variación presente
✓ Centroide v4 guardado


In [9]:
from scipy.stats import pearsonr
centroide_cidh = np.load(f"{REF_DIR}/centroide_cidh_v3.npy")

# Muestra de bloques del corpus
textos_muestra = []
import os
for f in sorted(os.listdir(CORPUS_C))[:10]:
    if f.endswith('.txt'):
        with open(f"{CORPUS_C}/{f}", encoding='utf-8') as fh:
            texto = fh.read()
        parrafos = [p.strip() for p in texto.split('\n\n') if 100 <= len(p.strip()) <= 500]
        textos_muestra.extend(parrafos[:5])

d_maf, d_cid = [], []
for t in textos_muestra[:50]:
    e = get_emb(t)
    d_maf.append(cosine_dist(e, centroide_v4))
    d_cid.append(cosine_dist(e, centroide_cidh))

r, p = pearsonr(d_maf, d_cid)
print(f"Correlación y₈/y₉ con centroide v4: r={r:.4f}, p={p:.4f}")
print(f"Meta: r < 0.80 | {'✓ alcanzada' if r < 0.80 else '⚠ pendiente'}")

ValueError: `x` and `y` must have length at least 2.

In [10]:
from scipy.stats import pearsonr

centroide_cidh = np.load(f"{REF_DIR}/centroide_cidh_v3.npy")

# Usar los candidatos ya cargados como muestra
textos_muestra = textos_nuevos[:50]

d_maf, d_cid = [], []
for t in textos_muestra:
    e = get_emb(t)
    d_maf.append(cosine_dist(e, centroide_v4))
    d_cid.append(cosine_dist(e, centroide_cidh))

r, p = pearsonr(d_maf, d_cid)
print(f"Correlación y₈/y₉ con centroide v4: r={r:.4f}, p={p:.4f}")
print(f"Meta: r < 0.80 | {'✓ alcanzada' if r < 0.80 else '⚠ pendiente'}")

# Distancia promedio de los candidatos al centroide v4
import numpy as np
dist_promedio = sum(d_maf) / len(d_maf)
print(f"\nDistancia promedio candidatos → centroide v4: {dist_promedio:.4f}")
print(f"(Menor = candidatos más cerca del polo MAFAPO)")

Correlación y₈/y₉ con centroide v4: r=0.6378, p=0.0000
Meta: r < 0.80 | ✓ alcanzada

Distancia promedio candidatos → centroide v4: 0.1210
(Menor = candidatos más cerca del polo MAFAPO)


In [12]:
import json
from datetime import datetime

inventario = {
    "version": "v4",
    "timestamp": datetime.now().isoformat(),
    "total_textos": 169,
    "componentes": {"v3b": 67, "candidatos_nuevos": 102},
    "margen_error_pct": 7.7,
    "dist_v3b_v4": 0.0066,
    "correlacion_y8_y9": float(r),
    "dist_promedio_candidatos": float(dist_promedio),
}
with open(f"{REF_DIR}/inventario_centroide_v4.json", 'w', encoding='utf-8') as f:
    json.dump(inventario, f, ensure_ascii=False, indent=2)
print("✓ Inventario guardado")

✓ Inventario guardado
